<img src="https://github.com/CropEvol/lecture/blob/master/textbook_2026/images/logo.png?raw=true" alt="2026年度ゲノム情報解析入門" height="100px" align="middle">

<div align="right"><a href="https://github.com/CropEvol/lecture#section2">実習表ページに戻る</a></div>

QTL-seq解析
---

　前回、小さいデータセットで形質と連関のあるSNP箇所を目視で探しました。また、バルクシーケンシングについても勉強しました。しかし、実際のゲノム配列は、非常に長く、目視で形質と連関のあるSNP箇所を探すのはほぼ不可能です。

　今回、バルクシーケンシングによるQTL解析法（遺伝子マッピング法）のひとつ**QTL-seq解析**を学び、SNP情報をどのように処理すればゲノムワイドなデータで遺伝子マッピングができるかを勉強していきます。



In [ ]:
## サンプルファイルのダウンロード
system("wget -q -O rice_qtlseq_dataset.csv https://github.com/CropEvol/lecture/raw/master/textbook_2025/dataset/rice_qtlseq_dataset.csv")
list.files(pattern = "rice")

# 今回の勉強内容

0. 前回の復習
1. アライメント結果からSNPデータセットを作る
1. SNPデータセットの扱い方
  1. データセットを読み込む
  1. 列データを取り出す
  1. 列データで計算をする
  1. 任意の染色体のデータ行を取り出す
  1. リードの深さを確認する
1. グラフ描画
  1. グラフ描画の基礎
  1. 散布図を描く
  1. Sliding window解析
  1. 原因遺伝子の位置を探す
  1. 全染色体を一度に描く
  1. ウィンドウの大きさを変える
  1. 候補領域を絞り込む
1. 発展: どこからが本物のピーク？

　今回も、Slidoを使ったアンケートがあります。🗳 Slido の印があるところでは、先にSlidoで回答してから、次のコードセルを実行してください。


# 0. 前回の復習

　前回、イネのQTL-seq解析のリードアライメント結果をみました。

イネのQTL-seq解析の概要:
- いもち病への抵抗性品種と罹病性品種の交配分離集団
- バルクシーケンシング:
  - 抵抗性個体のバルク
  - 罹病性個体のバルク
- 参照配列: 罹病性品種のゲノム配列

  

<img src="https://github.com/CropEvol/lecture/blob/master/textbook_2022/images/qtlseq_takagi_etal_2013_L06.png?raw=true" alt="Rice QTL-seq example (Takagi et al., 2013)" height="250px" align="middle">



<small>※ このノートブックは R で動いているため、IGV（前回使った `igv_notebook` は Python のライブラリです）は表示できません。アライメント結果を見直したい人は、前回のノートブック（リードアライメント）の「2-2. 実例: イネのバルクシーケンシング解析」のコードセルを実行してください。</small>

### 前回の振り返り

　前回、自分で選んだ4個体のS-bulkについて、120番目のSNPの「品種Bの塩基の割合」を計算してもらいました。

　120番目は、形質の原因となる55番目から離れたSNPです。形質と関係がない場所なので、「品種Bの塩基の割合」は0.5付近になると期待されます。しかし実際には、選んだ個体によって値が大きくばらつきます。

　この「ばらつき」は、実際のQTL-seq解析のデータにもつきまといます。今回は、このばらつきの中から、形質と本当に関係のある場所をどう見つけ出すかを考えていきます。

　実際のゲノムワイドなデータは、SNP箇所の数が多く、目視でSNP箇所をひとつずつ確認するのも難しいです。

　実際のゲノム解析では、アライメントデータからSNP箇所のみの情報（アリルの個数など）を抽出したデータセットを作ることができます。そのデータセットを解析することで、さまざまなデータ解析（系統解析や遺伝子マッピングなど）をおこなえます。

　今回、バルクシーケンシング由来のSNPアリルの個数がまとめられたデータセット（以下、SNPデータセット）をRで解析し、**QTL-seq解析**（遺伝子マッピング手法の一種）をおこなっていきましょう。

　QTL-seq解析では、SNPアリルの割合を使って、原因遺伝子があるゲノム領域の探索をおこないます。そのSNPアリルの割合のことを、QTL-seq解析では**SNP-index**と呼んでいます。

<img src="https://github.com/CropEvol/lecture/blob/master/textbook_2022/images/qtlseq_data_analysis.png?raw=true" alt="qtlseq_data_analysis" height="250px" align="middle">


# 1. アライメント結果からSNPデータセットを作る

　アラインメントを実施すると、BAMファイルというものが作られました。どのリードがどの場所にアラインメントされたかの情報が入ったファイルです。

　このアラインメント結果から、各サンプルのSNPの遺伝子型がどうなっているかというデータを作成する必要があります。

<img src="https://github.com/CropEvol/lecture/blob/master/textbook_2026/images/alignment_to_dataset.png?raw=true" alt="alignment to SNP dataset" height="250px" align="middle">

　詳細は省きますが、アライメント結果であるBAMファイルからSNPデータセットを作るには、既存のソフトウェア（[bcftools](https://samtools.github.io/bcftools/bcftools.html)など）を用います。

# 2. SNPデータセットの扱い方

　ここからは、SNPデータセットを **R** で扱うための基礎を勉強していきます。今回のノートブックでは、コードセルで **Python ではなく R** が動きます。


- イネQTL-seq解析のSNPデータセット: [rice_qtlseq_dataset.csv](https://raw.githubusercontent.com/CropEvol/lecture/master/textbook_2025/dataset/rice_qtlseq_dataset.csv)

## 2-1. データセットを読み込む

　SNPデータセットは、1行あたり8個のデータがカンマ記号(`,`)で区切られた、テーブル状の構造をしています。カンマ区切りのテーブル状データセットは、R の `read.csv` 関数で読み込むことができ、その後、データ処理をおこなえるようになります。読み込んだ表のことを、**データフレーム**と呼びます。

```r
# データセットを読み込む
データフレーム変数 <- read.csv("ファイル名")
```

<small>※ R では、変数に値を入れるときに `<-` を使います（`=` でも動きます）。</small>

In [ ]:
## データセットの読み込み
df <- read.csv("rice_qtlseq_dataset.csv")
## 表示
df

　読み込んだデータセットの見方は下図のとおりです。

<img src="https://github.com/CropEvol/lecture/blob/master/textbook_2022/images/qtlseq_takagi_etal_2013_L06.png?raw=true" alt="Rice QTL-seq example (Takagi et al., 2013)" height="250px" align="middle">

<br>
<br>
<br>

<img src="https://github.com/CropEvol/lecture/blob/master/textbook_2022/images/pandas_dataframe.png?raw=true" alt="pandas_dataframe" height="400px" align="middle">

## 2-2. 列データを取り出す

　つぎに、1列分のデータの取り出し方を学びましょう。

<img src="https://github.com/CropEvol/lecture/blob/master/textbook_2022/images/extract_one_column.png?raw=true" alt="extract_one_column" height="200px" align="middle">

```r
# 列データを取り出す
データフレーム変数$列名
　　　　または
データフレーム変数[ ,"列名"]
```

In [ ]:
# 「Bulk1_REF」列を取り出す
df$Bulk1_REF

### 練習2-2
　`Bulk1_ALT`の列を取り出してください。

In [ ]:
## 練習2-2: 「Bulk1_ALT」列を取り出す

#### 解答例

In [ ]:
df$Bulk1_ALT
#または df[ ,"Bulk1_ALT"]

## 2-3. 列データで計算する

　ここで**SNP-index**という指標を計算してみます。

SNP-indexは、変異(ALT型)のSNPの塩基数 / 全塩基数で計算し、SNP毎に計算します。

今回のデータでBulk1のSNP-indexを計算する場合は、下の様な式になります。

<img src="https://github.com/CropEvol/lecture/blob/master/textbook_2022/images/calculate_snp_index_of_bulk1.png?raw=true" alt="calculate_snp_index" height="60px" align="middle">

　実際に今回のデータを使って、SNP-indexの算出をおこないましょう。列データと列データの要素同士の計算は、次のようなコードで簡単におこなえます。

例えば列同士の足し算だと下図のようになります。

```r
# 列データ同士を計算する（足し算の場合）
列Aのデータ + 列Bのデータ
# 新しいデータとして追加する
データフレーム変数$新しい列名 <- 列Aのデータ + 列Bのデータ
```

<img src="https://github.com/CropEvol/lecture/blob/master/textbook_2022/images/pandas_calc_columns.png?raw=true" alt="pandas_calc_columns" height="200px" align="middle">

割り算も同様です。

　では、SNP毎にBulk1のSNP-indexを計算してみましょう。

<img src="https://github.com/CropEvol/lecture/blob/master/textbook_2022/images/calculate_snp_index_of_bulk1.png?raw=true" alt="calculate_snp_index" height="60px" align="middle">

In [ ]:
## Bulk1のSNP-indexを計算し、その結果を新しい列「Bulk1_SNPindex」として追加する
df$Bulk1_SNPindex <- df$Bulk1_ALT / (df$Bulk1_REF + df$Bulk1_ALT)
df

Bulk1_SNPindexが1に近いということは、抵抗性グループの殆どの個体ではそのSNPに変異が入っていることを示しています。

逆に0に近い場合は、抵抗性グループの殆どの個体はそのSNPに変異が無い、ということになります。

抵抗性グループの殆どの個体で変異が入っている、つまりその変異が原因ではないか？と考えられます。

### 練習2-3
　Bulk2のSNP-indexも算出して、データフレーム`df`の新しい列`Bulk2_SNPindex`に追加してください。

<img src="https://github.com/CropEvol/lecture/blob/master/textbook_2022/images/calculate_snp_index_of_bulk2.png?raw=true" alt="calculate_snp_index" height="60px" align="middle">

In [ ]:
## 練習2-3: Bulk2のSNPindexを計算し、その結果を新しい列「Bulk2_SNPindex」として追加する
df$Bulk2_SNPindex <-
df

#### 解答例

In [ ]:
df$Bulk2_SNPindex <- df$Bulk2_ALT / (df$Bulk2_REF + df$Bulk2_ALT)
df

## 2-4. 任意の染色体のデータ行を取り出す

　「2-3. 列データで計算する」で、各バルクについて、ゲノム全域のSNP-indexを算出しました。

　このあと、ひとつの染色体（例えば、`chr01`）のSNP-indexのグラフを描くために、任意の染色体のデータを取り出す方法を学びましょう。

```r
## 条件に合ったデータ行を取り出す（条件の後ろの「,」を忘れないように）
データフレーム変数[条件, ]
## 例: データフレーム「df」から、列「CHROM」が"chr01"の行を取り出す
## 「==」は等しいかどうかを比較する演算子
df[df$CHROM == "chr01", ]
```

<small>※ R のデータフレームでは `[行の指定, 列の指定]` の順に書きます。列の指定を空にすると、すべての列を取り出します。</small>

<img src="https://github.com/CropEvol/lecture/blob/master/textbook_2022/images/pandas_filtering_data.png?raw=true" alt="pandas_filtering_data" height="200px" align="middle">

In [ ]:
sub1 <- df[df$CHROM == "chr01", ]
sub1

### 練習2-4
　データフレーム「df」から、「CHROM」列がchr12のデータを取り出してください。

In [ ]:
## 練習2-4: dfから「CHROM」列がchr12のデータのみを取り出す
sub12 <-
sub12

#### 解答例

In [ ]:
sub12 <- df[df$CHROM == "chr12", ]
sub12

## 2-5. リードの深さを確認する

　SNP-indexは、リードの本数の割合です。前回の「1-7. ジェノタイピングは本当に簡単か？」で見たように、リードの本数（**リードの深さ、Depth**）が少ないと、偶然の偏りが大きくなります。また、反対に**深さが極端に多いSNP**にも注意が必要です。ゲノム中に何コピーもある配列（リピート）では、別々の場所のリードが1か所に集まって並んでしまうことがあるからです。

　まず、各SNPのリードの深さを計算し、その分布をヒストグラムで確認しましょう。

In [ ]:
## 各バルクのリードの深さ（REFの本数 + ALTの本数）を計算する
df$Bulk1_depth <- df$Bulk1_REF + df$Bulk1_ALT
df$Bulk2_depth <- df$Bulk2_REF + df$Bulk2_ALT

## 深さの要約（最小、中央値、平均、最大など）
summary(df[, c("Bulk1_depth", "Bulk2_depth")])

## ヒストグラム（「infinite values」という警告が出ることがありますが、問題ありません）
library(ggplot2)
options(repr.plot.width = 10, repr.plot.height = 3)   # グラフの大きさ（横, 縦）
ggplot(df, aes(x = Bulk1_depth)) +
  geom_histogram(binwidth = 2, boundary = 0) +
  scale_y_log10() +                       # 縦軸を対数目盛りにする（数の少ない部分も見えるように）
  labs(x = "Depth of Bulk1", y = "Number of SNPs")

　ほとんどのSNPの深さは10〜20本程度ですが、ごく一部に100本を超えるSNPがあります。これらはリピート領域などの、信頼性の低いSNPの可能性があります。

　データ行を取り出す条件は、`&`（かつ）でつなぐと、複数の条件を組み合わせられます。

```r
## 例: 列Aが10以上 かつ 列Bが10以上 の行を取り出す
データフレーム変数[データフレーム変数$列A >= 10 & データフレーム変数$列B >= 10, ]
```

### 練習2-5
　データフレーム `df` から、`Bulk1_depth` が40以下 **かつ** `Bulk2_depth` が40以下の行を取り出し、`df_filt` という名前の変数に入れてください。

In [ ]:
## 練習2-5: 深さが40以下のSNPだけを取り出す
df_filt <-
print(paste("取り出す前のSNP数:", nrow(df)))
print(paste("取り出した後のSNP数:", nrow(df_filt)))

#### 解答例

In [ ]:
df_filt <- df[df$Bulk1_depth <= 40 & df$Bulk2_depth <= 40, ]
print(paste("取り出す前のSNP数:", nrow(df)))
print(paste("取り出した後のSNP数:", nrow(df_filt)))

# 3. グラフ描画

　次に計算したSNP-indexの情報をグラフにしていきましょう。

## 3-1. グラフ描画の基礎

　R でのグラフ描画の基礎は次のとおりです。今回は、`ggplot2` というグラフ描画用のパッケージを使います。
> グラフのパーツが描かれた透明な層（レイヤー）を複数枚かさねて、ひとつのグラフを描く

<img src="https://github.com/CropEvol/lecture/blob/master/textbook_2022/images/drawing_plot_basis.png?raw=true" alt="drawing_plot_basis" height="180px" align="middle">


　`ggplot2` では、レイヤー（グラフフィールド、散布図、折れ線、軸ラベルなど）を `+` 記号で足していって、1つのグラフを作ります。

```r
library(ggplot2)
g <- ggplot(データフレーム変数, aes(x = x軸に使う列名, y = y軸に使う列名)) # グラフにするデータを指定
g <- g + geom_point() # どんな図を描くか(今回は散布図)を指定
g <- g + labs(x = "x軸ラベル", y = "y軸ラベル") # 軸ラベルを編集
g
```

## 3-2. 散布図を描く

　`ggplot2` を使って、SNP-indexの散布図を描いてみましょう。散布図のレイヤーは `geom_point()` です。

```r
# 散布図
g <- ggplot(データフレーム変数, aes(x = x座標に使う列, y = y座標に使う列)) +
g <- g + geom_point()
```

　ここでの例として、データフレーム`sub1`の情報（1番染色体の情報）を使って、Bulk1のSNP-indexの散布図を描きます。

- x軸座標データ: `POS`列
- y軸座標データ: `Bulk1_SNPindex`列

In [ ]:
library(ggplot2)
options(repr.plot.width = 12, repr.plot.height = 4) # グラフの大きさを指定（横, 縦）

g <- ggplot(sub1, aes(x = POS, y = Bulk1_SNPindex)) # グラフにするデータを指定
g <- g + geom_point() # 描くグラフとして散布図を指定
g <- g + labs(x = "Position", y = "SNP index of Bulk1") # 軸ラベルの編集
g

### 練習3-2

　「編集箇所1」と「編集箇所2」にコードを追記して、Bulk2のSNP-indexの散布図を描いてください。

In [ ]:
## 練習3-2: Bulk2のSNP-indexの散布図を描く
options(repr.plot.width = 12, repr.plot.height = 4)
## ------- 編集箇所1（y軸に使う列名を書く） ------------
g <- ggplot(sub1, aes(x = POS, y = ))
## ------- 編集箇所2（散布図のレイヤーを書く） ----------
g <-
## -----------------------------------------------------
g <- g + labs(x = "Position (Mbp)", y = "SNP index of Bulk2")
g

#### 解答例

In [ ]:
## 練習3-2: Bulk2のSNP-indexの散布図を描く
options(repr.plot.width = 12, repr.plot.height = 4)
## ------- 編集箇所1（y軸に使う列名を書く） ------------
g <- ggplot(sub, aes(x = POS, y = Bulk2_SNPindex))
## ------- 編集箇所2（散布図のレイヤーを書く） ----------
g <- g + geom_point()
## -----------------------------------------------------
g <- g + labs(x = "Position (Mbp)", y = "SNP index of Bulk2")
g

## 3-3. Sliding window解析

　散布図により各SNP箇所のSNP-indexを描画できましたが、そのグラフだけで原因遺伝子がありそうな場所を判断するのは難しそうです。

　バルクシーケンシングで得られるリアルなデータ（SNPアリルの個数、また、そこから求められるSNP-index）は、必ず期待値どおりの値が得られるわけではありません。ALT塩基を50%で得られることが期待できる場所（SNP-indexが0.5になる場所）では、その期待値周辺のSNP-indexが得られますが、ときには大きく外れたSNP-indexが得られることもあります。

<small>※ コイントスを例にすると、「裏面が出る確率50%のコインを10枚投げて、裏面が出た枚数を調べる」という何度も繰り返すと、裏が5枚出る試行が多く観察されますが、10枚とも裏が出たり、裏が0枚の試行も生じることがあります。SNP-indexも同じように、多くのSNPを得ると、期待値付近のSNP-indexが多く得られますが、大きく外れたSNP-indexも得られることがあります。</small>


　期待されるSNP-indexは、周辺のSNP座の平均SNP-indexで近似できます。任意の区間を設定し、その区間の平均SNP-indexを調べるといった作業を、ゲノム全域にわたっておこなうと、平均SNP-indexが高い場所や低い場所を見つけることができます。

　ゲノム全域にわたってSNP-indexの平均値の推移（**移動平均**）を調べる解析法を**Sliding window解析**と言います。

<img src="https://github.com/CropEvol/lecture/blob/master/textbook_2022/images/sliding_window_overview.png?raw=true" alt="sliding_window_overview" height="350px" align="middle">

　R では、`stats::filter` 関数を使うと移動平均を計算できます。少し書き方が複雑なので、次のコードセルで `rolling_mean` という関数を作っておきます。

```r
# 任意SNP数区間の移動平均を求める
rolling_mean(列データ, データ数)
```

<small>※ `rolling_mean(x, 20)` は、各SNPについて「そのSNPと、その前の19個のSNP」の合計20個の平均を計算します。最初の19個のSNPは、データが足りないので `NA`（値なし）になります。</small>

In [ ]:
## 移動平均を計算する関数を作る（このセルを一度実行してください）
rolling_mean <- function(x, n) {
  as.numeric(stats::filter(x, rep(1 / n, n), sides = 1))
}


In [ ]:
## Bulk1_SNPindexの移動平均
rolling_mean(sub$Bulk1_SNPindex, 20)  # Bulk1_SNPindex列（SNP 20個分）の移動平均

　また、移動平均の折れ線グラフは、折れ線のレイヤー `geom_line()` を重ねることで描けます。移動平均の値は、あらかじめデータフレームの新しい列に入れておきます。
```r
# 移動平均の折れ線グラフを追加する
g <- g + ggplot(データフレーム変数, aes(x = x座標に使う列, y = y座標に使う列))
g <- g + geom_point() # 散布図
g <- g + geom_line(aes(x = 位置の移動平均の列, y = 移動平均の列), オプション（任意）) # 移動平均の折れ線
g
```

In [ ]:
## SNPポジション、Bulk1_SNPindexの移動平均
sub1$win_x  <- rolling_mean(sub1$POS, 20)             # POS列（SNP 20個分）の移動平均
sub1$win_y1 <- rolling_mean(sub1$Bulk1_SNPindex, 20)  # Bulk1_SNPindex列（SNP 20個分）の移動平均

## 移動平均をグラフ化する
options(repr.plot.width = 12, repr.plot.height = 3)

g <- ggplot(sub1, aes(x = POS / 1000000, y = Bulk1_SNPindex))
g <- g + geom_point() # 散布図
g <- g + geom_line(aes(x = win_x / 1000000, y = win_y1), color = "orange", linewidth = 1, na.rm = TRUE)  # 移動平均の線グラフ
g <- g + labs(x = "Position (Mbp)", y = "SNP index of Bulk1") # 軸ラベル
g

これで単独のSNPのSNP_indexだけでなく、領域毎にSNP_indexの平均値を見る事が出来ます。

連鎖を考えると、原因遺伝子の領域で平均をとってもSNP_indexは高いままで、逆にシーケンスエラー等で偶然SNP_indexが高くなったSNP付近では低くなることが想定できます。

### 練習3-3

　`sub$win_y2 <-` の行に、Bulk2_SNPindexの移動平均を調べるコードを追記し、グラフに描いてください。

In [ ]:
## SNPポジション、Bulk2_SNPindexの移動平均
sub1$win_x  <- rolling_mean(sub1$POS, 20)   # POS列（SNP 20個分）の移動平均
sub1$win_y2 <-                              # Bulk2_SNPindex列（SNP 20個分）の移動平均

## 移動平均をグラフ化する
options(repr.plot.width = 12, repr.plot.height = 3)
g <- ggplot(sub1, aes(x = POS / 1000000, y = Bulk2_SNPindex))
g <- g + geom_point()
g <- g + geom_line(aes(x = win_x / 1000000, y = win_y2), color = "orange", linewidth = 1, na.rm = TRUE)
g <- g + labs(x = "Position (Mbp)", y = "SNP index of Bulk2")
g

#### 解答例

In [ ]:
## SNPポジション、Bulk2_SNPindexの移動平均
sub1$win_x  <- rolling_mean(sub1$POS, 20)   # POS列（SNP 20個分）の移動平均
sub1$win_y2 <- rolling_mean(sub1$Bulk2_SNPindex, 20)  # Bulk2_SNPindex列（SNP 20個分）の移動平均

## 移動平均をグラフ化する
options(repr.plot.width = 12, repr.plot.height = 3)
g <- ggplot(sub1, aes(x = POS / 1000000, y = Bulk2_SNPindex))
g <- g + geom_point()
g <- g + geom_line(aes(x = win_x / 1000000, y = win_y2), color = "orange", linewidth = 1, na.rm = TRUE)
g <- g + labs(x = "Position (Mbp)", y = "SNP index of Bulk2")
g

## 3-4. 原因遺伝子の位置を探す

今回、抵抗性グループのBulk1と罹病性グループのBulk2のそれぞれのSNP_indexを計算しました。

抵抗性のグループ(Bulk1)でSNP_indexが1に近いSNPの近くに原因遺伝子があると考えられはするのですが、そのSNPが罹病性のグループ(Bulk2)でもSNP_indexが高い場合、それは原因遺伝子ではない可能性が高くなります。

抵抗性のグループ(Bulk1)でSNP_indexが1に近く(=変異が入っている)、罹病性では0に近い(変異が無い)SNPがより原因遺伝子の可能性が高そうです。

そこで、Bulk1とBulk2のSNP_indexの差**Δ（デルタ）SNP-index**（Bulk1のSNP-index − Bulk2のSNP-index）を計算します。このΔSNP-indexが1または-1に近いほど原因遺伝子に近いSNPである可能性が高くなります。

<img src="https://github.com/slt666666/FAO_lecture/blob/main/FAO_2024/QTLseq_7.png?raw=true" alt="title" height="450px">


　イネには第1〜12染色体があります。次の二つのコードセルのプログラムを実行すると、任意の（二つ目のコードセルの`chrom`で指定した）染色体のSNP-indexをグラフ描画できます。それぞれの染色体のSNP-indexグラフを調べて、原因遺伝子がありそうな場所を探してください。

解析しているデータ：
- 参照配列: 罹病性イネ品種のゲノム配列
- 分離集団のバルク構成:
  - Bulk1: 抵抗性個体を集めたグループ
  - Bulk2: 罹病性個体を集めたグループ
- SNP-index: 抵抗性品種由来のSNPアリルの割合


In [ ]:
## グラフ描画用の関数
library(ggplot2)
draw_QTLseq <- function(pos, bulk1, bulk2, delta, window_size = 20) {
  ## グラフ用データ（3つのグラフ分を縦に並べた表を作る）
  panels <- c("SNP-index of Bulk1", "SNP-index of Bulk2", "delta SNP-index")
  d <- data.frame(x = pos / 1000000,                                    # x座標データ: POS (x Mbp)
                  y = c(bulk1, bulk2, delta),                           # y座標データ
                  panel = factor(rep(panels, each = length(pos)), levels = panels))
  d$win_x <- rolling_mean(pos / 1000000, window_size)                 # POS移動平均
  d$win_y <- c(rolling_mean(bulk1, window_size),                       # 各グラフのSNP-index移動平均
               rolling_mean(bulk2, window_size),
               rolling_mean(delta, window_size))
  ## 縦軸の範囲（Bulk1, Bulk2 は -0.1〜1.1、delta は -1.1〜1.1）
  limits <- data.frame(panel = factor(rep(panels, each = 2), levels = panels),
                       x = min(d$x), y = c(-0.1, 1.1, -0.1, 1.1, -1.1, 1.1))
  ## グラフ描画
  options(repr.plot.width = 12, repr.plot.height = 8)
  g <- ggplot(d, aes(x = x, y = y)) +
    geom_point(aes(color = panel), alpha = 0.3) +
    geom_line(aes(x = win_x, y = win_y), color = "black", linewidth = 1.5, na.rm = TRUE) +
    geom_blank(data = limits) +
    facet_grid(panel ~ ., scales = "free_y") +
    labs(x = "Position (x Mbp)", y = NULL) +
    theme(legend.position = "none", text = element_text(size = 18))
  plot(g)
}

In [ ]:
## 表示する染色体番号（1〜12の数字を入力可能）
chrom <- 1

## ---------------------------
## データセット
df2  <- read.csv("rice_qtlseq_dataset.csv")
sub2 <- df2[df2$CHROM == sprintf("chr%02d", chrom), ]
## SNP-index算出
position    <- sub2$POS
bulk1_index <- sub2$Bulk1_ALT / (sub2$Bulk1_REF + sub2$Bulk1_ALT)
bulk2_index <- sub2$Bulk2_ALT / (sub2$Bulk2_REF + sub2$Bulk2_ALT)
delta_index <- bulk1_index - bulk2_index
## グラフ描画
draw_QTLseq(position, bulk1_index, bulk2_index, delta_index)

## 3-5. 全染色体を一度に描く

　3-4 では、染色体を1本ずつ表示しました。`for` 文（繰り返し）を使うと、12本の染色体のグラフを一度に描けます。ここでは、2つのバルクの差である **Δ（デルタ）SNP-index**（Bulk1のSNP-index − Bulk2のSNP-index）の移動平均を描きます。

```r
## for文の例: 1から12までの数字を順に chrom に入れて、繰り返し処理する
for (chrom in 1:12) {
  print(chrom)
}
```

　ここでは、`for` 文で染色体ごとに移動平均を計算し、`ggplot2` の `facet_grid` で、12本の染色体のグラフを縦に並べて描きます。

　Δ SNP-indexは、2つのバルクのアリルの割合に差がない場所では0付近になり、差が大きい場所ほど1（または−1）に近づきます。

In [ ]:
window <- 20   # 移動平均をとるSNPの数

## Δ SNP-indexを計算する
df$Bulk1_SNPindex <- df$Bulk1_ALT / df$Bulk1_depth
df$Bulk2_SNPindex <- df$Bulk2_ALT / df$Bulk2_depth
df$delta <- df$Bulk1_SNPindex - df$Bulk2_SNPindex

## 染色体ごとに移動平均を計算する
df$win_x <- NA
df$win_delta <- NA
for (chrom in 1:12) {
  rows <- df$CHROM == sprintf("chr%02d", chrom)                 # chrom番目の染色体の行
  df$win_x[rows]     <- rolling_mean(df$POS[rows], window)
  df$win_delta[rows] <- rolling_mean(df$delta[rows], window)
}

## 12本の染色体を縦に並べて描く
options(repr.plot.width = 12, repr.plot.height = 18)
ggplot(df, aes(x = POS / 1000000, y = delta)) +
  geom_point(size = 0.5, alpha = 0.2, color = "gray") +
  geom_line(aes(x = win_x / 1000000, y = win_delta), color = "red", linewidth = 1, na.rm = TRUE) +
  geom_hline(yintercept = 0, linewidth = 0.3) +
  facet_grid(CHROM ~ .) +
  ylim(-1.1, 1.1) +
  labs(x = "Position (Mbp)", y = "delta SNP-index")

### 原因遺伝子座

　第6染色体の3~5Mbのあたりに、バルク間で平均SNP-indexに大きな差がみられます。そのゲノム領域にいもち病への抵抗性-罹病性に関係する遺伝子がありそうです。

## 3-6. ウィンドウの大きさを変える

　移動平均をとるSNPの数（**ウィンドウの大きさ**）は、解析する人が決めます。ウィンドウの大きさを変えると、グラフはどのように変わるでしょうか？

### 練習3-6
　次のコードセルの `windows` に、ウィンドウの大きさ（SNPの数）を3つ入れて実行してください。まずは `c(5, 20, 100)` を試し、その後、自由に変えてみましょう。

In [ ]:
windows <- c(5, 20, 100)   # 比べるウィンドウの大きさ（SNPの数）

sub6 <- df[df$CHROM == "chr06", ]
lines <- data.frame()
for (w in windows) {
  win_x <- rolling_mean(sub6$POS, w) / 1000000
  win_y <- rolling_mean(sub6$delta, w)
  lines <- rbind(lines, data.frame(win_x = win_x, win_y = win_y, window = paste(w, "SNPs")))
  best <- which.max(win_y)
  print(sprintf("ウィンドウ %4d SNP: Δ SNP-indexが最大になる位置 = %.2f Mbp（最大値 %.2f）", w, win_x[best], win_y[best]))
}
lines$window <- factor(lines$window, levels = paste(windows, "SNPs"))

options(repr.plot.width = 12, repr.plot.height = 4)
ggplot(sub6, aes(x = POS / 1000000, y = delta)) +
  geom_point(size = 1, alpha = 0.2, color = "gray") +
  geom_line(data = lines, aes(x = win_x, y = win_y, color = window), linewidth = 1, na.rm = TRUE) +
  geom_hline(yintercept = 0, linewidth = 0.3) +
  labs(x = "Position (Mbp)", y = "delta SNP-index", color = "window")

### 確認ポイント
- ウィンドウを小さくすると、線は安定しなくなり、ウィンドウを大きくすると、線は安定しますが、大雑把になってしまいます。
- Δ SNP-indexが最大になる位置も、ウィンドウの大きさによって少しズレることになります。
- ウィンドウを大きくしすぎると、確かに線は安定しますが、原因遺伝子の場所が絞り込みづらくなります。

## 3-7. 候補領域を絞り込む

　グラフを目で見るだけでなく、条件を使って、Δ SNP-indexが大きい領域の範囲を数値で取り出してみましょう。ここでは、ウィンドウ20 SNPの移動平均が0.5を超える範囲を、候補領域とします。

In [ ]:
window    <- 20    # ウィンドウの大きさ
threshold <- 0.5   # Δ SNP-indexの基準値

sub6  <- df[df$CHROM == "chr06", ]
win_x <- rolling_mean(sub6$POS, window)
win_y <- rolling_mean(sub6$delta, window)

## 基準値を超えるウィンドウの位置を取り出す
high <- win_x[which(win_y > threshold)]
print(sprintf("Δ SNP-indexの移動平均が %.1f を超える範囲: chr06 の %.2f 〜 %.2f Mbp", threshold, min(high) / 1e6, max(high) / 1e6))

　QTL-seq解析で候補領域が見つかったら、次は、その領域の中にどんな遺伝子があるかを調べ、原因遺伝子の候補を絞り込んでいきます。

###　確認ポイント
　イネのゲノムデータベース [RAP-DB](https://rapdb.dna.affrc.go.jp/) の「Genome Browser」で、上で求めた範囲（例: `chr06:2000000..5000000`）を表示し、どんな遺伝子があるか見てみましょう。遺伝子の名前や説明に、病害抵抗性に関係しそうな言葉（disease resistance、NBS-LRR など）が含まれる遺伝子はありますか？

<small>※ 今回のデータで使った参照配列（罹病性品種のゲノム配列）と、RAP-DBの参照配列（日本晴）では、位置が少しずれている可能性があります。おおよその目安として調べてください。</small>

---
# 4. 発展: どこからが本物のピーク？

　3-5 の全染色体のグラフをもう一度見てください。染色体6のピークは明らかですが、ほかの染色体にも、Δ SNP-indexが0.3〜0.5くらいまで振れている場所があります。たとえば、染色体9です。

このあたりの領域は「本物の」原因遺伝子があると考えてよいのでしょうか？

　「本物」かどうかを判断するために、**もし形質と関係する遺伝子がまったくなかったら、Δ SNP-indexはどのくらいまで振れるのか**を、シミュレーションで調べてみましょう。

　遺伝子と関係のない場所では、どちらのバルクでもSNP-indexの期待値は0.5です。そこで、各SNPについて、実際のデータと同じリードの深さで「50%の確率でALT」というコイントスをおこない、偽物のSNP-indexを作ります。これを100回繰り返して、移動平均のΔ SNP-indexが偶然でどこまで振れるか（99%の範囲）を求めます。

In [ ]:
window <- 20
set.seed(1)   # 乱数の種（同じ結果を再現できるようにする）
d1 <- df$Bulk1_depth
d2 <- df$Bulk2_depth

limits <- c()
for (t in 1:100) {
  ## 各SNPで「深さの本数だけコイントスし、ALTが出た割合」を計算する
  sim1 <- rbinom(length(d1), size = d1, prob = 0.5) / d1
  sim2 <- rbinom(length(d2), size = d2, prob = 0.5) / d2
  ## 染色体ごとに移動平均をとる
  sim_win <- ave(sim1 - sim2, df$CHROM, FUN = function(v) rolling_mean(v, window))
  ## 偶然で生じる振れの99%の範囲
  limits <- c(limits, quantile(abs(sim_win), 0.99, na.rm = TRUE))
}
limit <- mean(limits)
print(sprintf("リードの読み取りの偶然だけで生じる Δ SNP-index の振れ（99%%の範囲）: ±%.3f", limit))

## 実際のデータで、各染色体の移動平均の最大の振れと比べる
obs <- tapply(df$delta, df$CHROM, function(v) max(abs(rolling_mean(v, window)), na.rm = TRUE))
data.frame(最大の振れ = round(obs, 2), 範囲を超えた = obs > limit)

　染色体9のΔ SNP-indexは0.5前後でしたが、これはギリギリ偶然だけで生じうるΔ SNP-indexの振れ幅に含まれます。つまり、染色体9に原因遺伝子が無くても起きうる値と判断できます。

しかし、効果が小さい遺伝子がある場合も十分考えられるので、

- 別の個体を選んでバルクを作り直し、同じ場所に振れが再現するかを見る
- バルクに入れる個体数を増やす

などの追加アプローチが必要だと考えられるところです。

---
# まとめ

　このテキストでは、Rを使って、QTL-seq解析をおこないました。

このテキストで勉強した内容：
- RでSNPデータセットを読み込む
- SNPアリルの個数からSNP-indexを算出する
- SNP-indexの散布図を描く
- SNP-indexの移動平均の折れ線グラフを描き、原因遺伝子の位置を探索する
- リードの深さで信頼性の低いSNPを除く
- for文で全染色体のグラフを描く
- ウィンドウの大きさと、候補領域の絞り込み
- 偶然のばらつきと「本物のピーク」の見極め

　ここ三回の実習をとおして、ゲノム解析の基礎（ゲノムアセンブル、リードアライメント、SNP検出）を勉強してきました。また、実際に使われているゲノム解析の一例として、QTL-seq解析も勉強しました。

　次回以降、**Genome-Wide Association Study (GWAS)** や **Genomic prediction** といったゲノム解析を勉強していきます。


<div align="right"><a href="https://github.com/CropEvol/lecture#section2">実習表ページに戻る</a></div>